# Line-up photos: who is who in red, from the walk-out

Before kick-off the teams walk out close to the camera (around 6:20-7:40 of the original
`match_video_2.mp4`), where the shirt numbers are readable. This notebook tracks the players in that minute,
takes many photos of each, and computes their appearance with the same model used for the match pieces.
You then tell Claude which marked player wears which number; those photos become the reference look of
each red player, and the match pieces are named from them.

Output in `MyDrive/Playbook/review/match_video_2/lineup/`: `lineup_ids.jpg` (one picture, every player
marked with a number), `lineup_cards.jpg` + `lineup_cards.json` (photos per player), `lineup_emb.npz`.

**Runtime:** L4 GPU, about 5 minutes. **Runtime -> Run all**; do not edit cells.

In [ ]:
# Cell 1 — GPU, Drive, the original video
import os, json, time, glob, shutil, subprocess, sys
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> L4 GPU, then re-run.')
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
START, END = '00:06:20', '00:07:40'      # the walk-out in the original video
found = [p for p in [f'{PBD}/video_downloads/match_video_2.mp4'] if os.path.exists(p)] or \
        glob.glob(f'{PBD}/**/match_video_2.mp4', recursive=True)
if not found:
    raise SystemExit('match_video_2.mp4 (the original, full download) not found under MyDrive/Playbook.')
ORIGINAL = found[0]
MODEL = f'{PBD}/reid_appearance/appearance_model.pt'
OUT = f'{PBD}/review/match_video_2/lineup'; os.makedirs(OUT, exist_ok=True)
print('original:', ORIGINAL)

In [ ]:
# Cell 2 — Our repo and packages
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core', 'geometry')]:
    del sys.modules[m]
os.chdir(PB); sys.path.insert(0, PB)
!pip install -q "rfdetr[train]==1.11.0" "git+https://github.com/roboflow/sports.git@main" "pydantic-settings==2.4.0" "python-dotenv==1.0.1" "lap>=0.5.13,<0.6" "transformers>=4.47"
shutil.copy('baseline.env', '.env')
import numpy as np, pandas as pd, cv2, torch
print('ready')

In [ ]:
# Cell 3 — Cut the walk-out and track the players in it (~3 min)
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-ss', START, '-to', END, '-i', ORIGINAL, '-an',
                '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '14', '/content/lineup.mp4'], check=True)
r = subprocess.run([sys.executable, 'main.py', '--source-video', '/content/lineup.mp4', '--out-dir', '/content/lineup',
                    '--no-video', '--enable-team'], capture_output=True, text=True,
                   env={**os.environ, 'MODEL_ACCEL': 'fp16', 'BALL_MODEL_ACCEL': 'fp16'})
if r.returncode != 0:
    print(r.stdout[-1500:]); print(r.stderr[-3000:]); raise SystemExit('tracking failed: send Claude the lines above')
t = pd.read_csv('/content/lineup/per_frame_tracks.csv', low_memory=False)
t = t[t.class_id.isin((1, 2, 3)) & (t.raw_tracker_id >= 0)]
print(len(t), 'boxes,', t.raw_tracker_id.nunique(), 'tracks')

In [ ]:
# Cell 4 — Photos and appearance of every track, and one picture with every player marked
from tools.reid_appearance import Embedder, crops_from_video, sample_rows
long = t.groupby('raw_tracker_id').filter(lambda g: len(g) >= 15)
rows = sample_rows(long, 'raw_tracker_id', per_id=24)
crops, rows = crops_from_video('/content/lineup.mp4', rows)
emb = Embedder(); emb.load_state_dict(torch.load(MODEL, map_location=emb.device))
E = emb.embed(crops)
np.savez_compressed(f'{OUT}/lineup_emb.npz', emb=E.astype(np.float16), track=rows.raw_tracker_id.to_numpy(),
                    frame=rows.frame.to_numpy(), h=(rows.y2 - rows.y1).to_numpy())
# per track: 6 photos (its biggest boxes), 64x128 each
W, H, K = 64, 128, 6
tracks = sorted(rows.raw_tracker_id.unique())
sheet = np.full((max(1, (len(tracks) + 1) // 2) * H, 2 * K * W, 3), 24, np.uint8); index = {}
for i, tr in enumerate(tracks):
    m = np.where(rows.raw_tracker_id.to_numpy() == tr)[0]
    m = m[np.argsort(-(rows.y2 - rows.y1).to_numpy()[m])][:K]
    x0, y0 = (i % 2) * K * W, (i // 2) * H
    for k, j in enumerate(m):
        sheet[y0:y0 + H, x0 + k * W:x0 + (k + 1) * W] = cv2.resize(crops[j], (W, H))[:, :, ::-1]
    cv2.putText(sheet, str(tr), (x0 + 2, y0 + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
    index[str(int(tr))] = [x0, y0, len(m)]
cv2.imwrite(f'{OUT}/lineup_cards.jpg', sheet); json.dump({'w': W, 'h': H, 'tracks': index}, open(f'{OUT}/lineup_cards.json', 'w'))
# the frame where most tracks are on screen, every box marked with its track number
best = t.groupby('frame').raw_tracker_id.nunique().idxmax()
cap = cv2.VideoCapture('/content/lineup.mp4'); cap.set(cv2.CAP_PROP_POS_FRAMES, int(best)); ok, fr = cap.read(); cap.release()
for r_ in t[t.frame == best].itertuples():
    col = (0, 0, 255) if r_.class_id == 3 else (0, 255, 255)
    cv2.rectangle(fr, (int(r_.x1), int(r_.y1)), (int(r_.x2), int(r_.y2)), col, 2)
    cv2.putText(fr, str(int(r_.raw_tracker_id)), (int(r_.x2) + 3, int(r_.y1) + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)
cv2.imwrite(f'{OUT}/lineup_ids.jpg', fr)
print(len(tracks), 'players/officials with photos; picture frame', best)
print(f'Done. Tell Claude the line-up photos are in {OUT}.')